# Fig 6C and supplementary power-drop loci — AD GWAS loci where SuSiE-rRSS and SuSiE-RSS differ (data processing)

For each selected block × study: SuSiE-RSS and SuSiE-rRSS PIPs / CSs, GWAS −log10 p, and AD xQTL evidence (cV2F, inclusion score, colocalization VCP), reduced by `load_locus()` in `R/utils.R`.

Two separate sets of loci:
1. **Fig 6C** — CR1, MS4A4A, CASS4.
2. **Supplementary (Fig S12c): SuSiE-rRSS power drop** — loci where SuSiE-rRSS reports no credible set at a signal SuSiE-RSS resolves, with SuSiE-rRSS's PIP sum over each missed set. Kept apart from the Fig 6C examples.

**Outputs** (named lists of loci; each = `list(pip_df, ev, ss, reg, bid, sid)`):
- `figure_data/Fig6C.AD_GWAS_Differing_Loci.rds` — Fig 6C loci
- `figure_data/FigS13C.AD_GWAS_Additional_Loci.rds` — power-drop loci
- `figure_data/FigS13C.AD_GWAS_Power_Drop_CS_PIP_Support.csv` — power-drop loci: each SuSiE-RSS set, SuSiE-rRSS's PIP sum over its variants and whether SuSiE-rRSS reports a set there

In [1]:
source("../../R/utils.R")
library(pecotmr)
devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)   # getSumStatsDf()
library(readxl)
filter <- dplyr::filter; select <- dplyr::select

ℹ Loading pecotmr
Creating a generic function for ‘endoapply’ from package ‘S4Vectors’ in package ‘pecotmr’

Warning message:
“Failed to load at least one DLL.
Caused by error in `getDLLRegisteredRoutines.DLLInfo()`:
! must specify DLL via a “DLLInfo” object. See getLoadedDLLs()”


In [2]:
### Load AD GWAS explorer data
AD_loci <- suppressMessages(
  suppressWarnings(
    read_excel(
      "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/unified_AD_loci_xQTL_summary.xlsx",
      skip = 2
    )
  )
)

cs_counts_df <- readRDS("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export//method_compare.cs_counts.rds")

## blocks where the number of CSs differs between methods (no APOE, pre-2026 GWAS)
region_diff <- cs_counts_df %>% filter(diff != 0) %>% filter(block_id != apoe_block) %>% filter(!str_detect(study, "2026"))

region_diff %>% pull(block_id) %>% unique() %>% length()                                           # 14 regions before 2026
cs_counts_df %>% filter(diff != 0) %>% filter(block_id != apoe_block) %>% pull(block_id) %>% unique() %>% length()   # 22 incl. 2026

[1] 14

[1] 22

## Fig 6D: CR1, MS4A4A, CASS4

In [3]:
main_idx <- c(CR1    = 8,    # chr1_205972031_208461272  / AD_Wightman_Full_2021
              MS4A4A = 2,    # chr11_56858541_63818332   / AD_Wightman_Excluding23andMe_2021
              CASS4  = 9)    # chr20_55439357_57610823   / AD_Wightman_Full_2021

loci_main <- map(main_idx, ~ load_locus(region_diff$block_id[.x], region_diff$study[.x], AD_loci))
map_chr(loci_main, ~ paste(.x$bid, .x$sid))

save_data(loci_main, "Fig6C.AD_GWAS_Differing_Loci")

CR1 
           "chr1_205972031_208461272 AD_Wightman_Full_2021" 
                                                     MS4A4A 
"chr11_56858541_63818332 AD_Wightman_Excluding23andMe_2021" 
                                                      CASS4 
            "chr20_55439357_57610823 AD_Wightman_Full_2021"

## Supplementary (Fig S12c): SuSiE-rRSS power drop

Loci where SuSiE-rRSS reports no credible set at a signal SuSiE-RSS resolves at 95% coverage. SuSiE-rRSS's PIPs summed over the variants of each missed SuSiE-RSS set show how narrowly it missed: a sum just below 0.95 is a set that falls short of the coverage rather than a lost signal. These can be picked up at a lower coverage or by colocalization.

In [4]:
power_drop_idx <- c(1,     # chr11_46916060_49936088   / AD_Wightman_Full_2021
              11,    # chr2_231326994_235132526  / AD_Bellenguez_EADB_2022
              6)     # chr19_53788025_55215821   / AD_Wightman_Excluding23andMe_2021

loci_power_drop <- map(power_drop_idx, ~ load_locus(region_diff$block_id[.x], region_diff$study[.x], AD_loci))
names(loci_power_drop) <- map_chr(loci_power_drop, ~ paste(.x$sid, .x$bid, sep = "."))
names(loci_power_drop)

save_data(loci_power_drop, "FigS13C.AD_GWAS_Additional_Loci")

[1] "AD_Wightman_Full_2021.chr11_46916060_49936088"            
[2] "AD_Bellenguez_EADB_2022.chr2_231326994_235132526"         
[3] "AD_Wightman_Excluding23andMe_2021.chr19_53788025_55215821"

In [5]:
## each SuSiE-RSS set at these loci: SuSiE-rRSS's PIP sum over its variants
cs_support <- map_dfr(loci_power_drop, cs_pip_support)
cs_support %>% filter(!other_in_cs) %>% as.data.frame()
write_csv(cs_support, file.path(data_dir, "FigS13C.AD_GWAS_Power_Drop_CS_PIP_Support.csv"))

sid,bid,method,cs,n_variants,pip_sum,other,other_pip_sum,other_in_cs
<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<chr>,<dbl>,<lgl>
AD_Wightman_Full_2021,chr11_46916060_49936088,SuSiE-RSS,1,16,0.9594047,SuSiE-rRSS,7.756830e-01,FALSE
AD_Wightman_Full_2021,chr11_46916060_49936088,SuSiE-RSS,2,206,0.9609717,SuSiE-rRSS,8.116184e-05,FALSE
AD_Bellenguez_EADB_2022,chr2_231326994_235132526,SuSiE-RSS,1,1,0.9532832,SuSiE-rRSS,9.443119e-01,FALSE
AD_Wightman_Excluding23andMe_2021,chr19_53788025_55215821,SuSiE-RSS,1,4,0.9628401,SuSiE-rRSS,9.294071e-01,FALSE
